In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window
spark = SparkSession.builder \
    .appName("scenario_3") \
    .master("local[*]") \
    .config("spark.driver.memory",              "6g") \
    .config("spark.executor.memory",            "6g") \
    .config("spark.sql.shuffle.partitions",     "50") \
    .config("spark.sql.adaptive.enabled",       "true") \
    .config("spark.sql.autoBroadcastJoinThreshold", "-1") \
    .config("spark.python.worker.reuse",        "false") \
    .config("spark.executor.heartbeatInterval", "60s") \
    .config("spark.network.timeout",            "300s") \
    .config("spark.hadoop.io.native.lib.available", "false") \
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
    .getOrCreate()

In [4]:
df=spark.read.format('parquet')\
   .option("inferschema",True)\
   .option("header",True)\
   .load("E:/Big_data/Ansh_lamba/Post/Post_3/pyspark_scenarios/dataset_generation/data/dataset2_user_events/events/v1/")

In [5]:
df_user=spark.read.format('parquet')\
        .option("inferschema",True)\
        .option("header",True)\
        .load("E:/Big_data/Ansh_lamba/Post/Post_3/pyspark_scenarios/dataset_generation/data/dataset2_user_events/user_profile")

In [6]:
df.show()

+-----------+--------+----------+----------+-------------------+------------------+----------------+-----------+-------+-------+-----------+
|   event_id| user_id|session_id|event_type|    event_timestamp|         page_name|duration_seconds|device_type|     os|country|app_version|
+-----------+--------+----------+----------+-------------------+------------------+----------------+-----------+-------+-------+-----------+
|E0000666708|U0000803| S00269219|     click|2024-01-01 00:00:08|           profile|            NULL|     mobile|    Mac|     US|        3.5|
|E0000429699|U0020459| S00498965|    logout|2024-01-01 00:00:11|order_confirmation|            NULL|     mobile|    iOS|     US|        4.0|
|E0000209181|U0007420| S00189644| page_view|2024-01-01 00:00:16|          checkout|           252.0|    desktop|Android| Brazil|        4.1|
|E0000112732|U0018428| S00206923|  purchase|2024-01-01 00:00:16|     category_page|           451.0|    desktop|Android|  India|        4.1|
|E0000550706|

In [7]:
df_user.show()

+--------+------------+-----------+------------+
| user_id|user_segment|signup_date|account_type|
+--------+------------+-----------+------------+
|U0000001|     Premium| 2020-03-22|  Individual|
|U0000002|    Standard| 2023-09-09|  Individual|
|U0000003|       Trial| 2023-06-17|  Enterprise|
|U0000004|       Trial| 2023-04-12|  Individual|
|U0000005|    Standard| 2021-07-02|  Individual|
|U0000006|    Standard| 2022-02-14|  Enterprise|
|U0000007|     Premium| 2020-04-07|  Individual|
|U0000008|    Standard| 2022-02-23|  Individual|
|U0000009|    Standard| 2022-12-11|  Individual|
|U0000010|     Premium| 2022-06-23|  Individual|
|U0000011|    Standard| 2020-03-25|  Individual|
|U0000012|    Standard| 2020-08-05|  Individual|
|U0000013|    Standard| 2023-04-10|  Enterprise|
|U0000014|    Standard| 2020-04-30|  Individual|
|U0000015|    Standard| 2023-04-30|  Individual|
|U0000016|    Standard| 2021-12-09|  Individual|
|U0000017|    Standard| 2023-10-24|  Individual|
|U0000018|    Standa

In [9]:
df1=df.join(df_user,'user_id','inner').limit(5)
df1.show()
df1.explain()

+--------+-----------+----------+----------+-------------------+---------------+----------------+-----------+-------+--------+-----------+------------+-----------+------------+
| user_id|   event_id|session_id|event_type|    event_timestamp|      page_name|duration_seconds|device_type|     os| country|app_version|user_segment|signup_date|account_type|
+--------+-----------+----------+----------+-------------------+---------------+----------------+-----------+-------+--------+-----------+------------+-----------+------------+
|U0000013|E0000636559| S00227900|     click|2024-01-01 02:48:08|product_listing|            NULL|    desktop|    iOS|      US|        3.2|    Standard| 2023-04-10|  Enterprise|
|U0000013|E0000525350| S00478887| page_view|2024-01-01 19:44:07|           home|           358.0|     mobile|Windows|Portugal|        4.1|    Standard| 2023-04-10|  Enterprise|
|U0000013|E0000208662| S00013152|     click|2024-01-02 06:30:30|           help|            NULL|    desktop|    iO

In [10]:
df2=df.join(broadcast(df_user),'user_id').orderBy('user_id').limit(5)
df2.show()
df2.explain()

+--------+-----------+----------+----------+-------------------+--------------+----------------+-----------+-------+-------+-----------+------------+-----------+------------+
| user_id|   event_id|session_id|event_type|    event_timestamp|     page_name|duration_seconds|device_type|     os|country|app_version|user_segment|signup_date|account_type|
+--------+-----------+----------+----------+-------------------+--------------+----------------+-----------+-------+-------+-----------+------------+-----------+------------+
|U0000001|E0001757205| S00271936|     error|2024-02-15 02:29:21| category_page|            NULL|    desktop|Windows|     US|        4.1|     Premium| 2020-03-22|  Individual|
|U0000001|E0004052314| S00065447|     click|2024-05-27 06:19:28|         login|            NULL|    desktop|Android| Brazil|        3.0|     Premium| 2020-03-22|  Individual|
|U0000001|E0000313797| S00155187| page_view|2024-01-05 00:37:04|         login|           502.0|     mobile|Android|     US| 